In [ ]:
import os
from tqdm import tqdm
from glob import glob

import numpy as np
import pandas as pd

from datasets import Dataset, DatasetDict, load_metric

from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments,
                          DataCollatorForSeq2Seq, Trainer)
from peft import LoraConfig, get_peft_model, TaskType

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level = logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

In [ ]:
class CFG:
    wandb = True
    report_to = None
    lab_assignment = 3
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    prefix_val = "summarize: "
    output_dir = "processed_data"
    model_save_dir = "PEFT_T5"

    tokenizer_name = "google/t5-efficient-mini"
    model_name = "google/t5-efficient-mini"

    project = 'NUM-Machine-Learning-Lab-3'
    name = "Lab 3 Model Training - Text Summarization T5 with ROUGE-1"

    config = {
        "output_dir": "t5_small_lab3_finetune_PEFT",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        'num_train_epochs': 3,
        "train_batch_size": 12,
        "eval_batch_size": 4,
        "max_seq_length": 1024,
        "overwrite_output_dir": True,
        "reprocess_input_data": True,
        "fp16": True
    }

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "rouge"

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

# 1. Өгөгдлөө модел сургахад бэлтгэх

In [ ]:
df_names = glob("../Lab 2/processed_dfs/*.parquet")
search_terms = '\n'.join([os.path.basename(filename).split('_')[1] for filename in df_names])

df = pd.DataFrame()
for df_name in tqdm(df_names):
    df = pd.concat([df, pd.read_parquet(df_name)[['title', 'abstract']]])

df.drop_duplicates(inplace = True)
df.reset_index(drop = True, inplace = True)

df.rename(columns = {'title': 'target_text', 'abstract': 'input_text'}, inplace = True)
df['prefix'] = CFG.prefix_val

os.makedirs(CFG.output_dir, exist_ok = True)
output_filename = os.path.join(CFG.output_dir, "arxiv_title_generation.parquet")
df.to_parquet(output_filename)

In [ ]:
print("\nDataframe memory usage")
print(df.memory_usage(deep = True))

print(f"Dataframe shape: {df.shape}\n")
print(f"All search terms:\n{search_terms}")

print(df.head())

In [ ]:
test_size = CFG.test_size
test_df = df.sample(frac = test_size, random_state = 1970)
train_df = df.drop(index = test_df.index)

print(f"Training instance count: {len(train_df)}\nTest instance count: {len(test_df)}\n")

train_dataset = Dataset.from_dict(train_df)
test_dataset = Dataset.from_dict(test_df)
arxiv_title_dict = DatasetDict({"train": train_dataset,"test": test_dataset})

output_filename = os.path.join(CFG.output_dir, "arxiv_title_generation_dataset")
arxiv_title_dict.save_to_disk(output_filename)

print(arxiv_title_dict)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name, use_fast = False)

def preprocess_function(examples):
    inputs = [CFG.prefix_val + doc for doc in examples["input_text"]]
    model_inputs = tokenizer(inputs,
                             max_length = config['max_seq_length'],
                             padding = True,
                             truncation = True)

    labels = tokenizer(text_target = examples["target_text"],
                       max_length = config["max_seq_length"] // 4,
                       padding = True,
                       truncation = True)
    
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

tokenized_arxiv = arxiv_title_dict.map(preprocess_function, batched = True)

In [ ]:
rouge = load_metric(CFG.eval_metric)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, 
                                           skip_special_tokens = True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(labels, 
                                            skip_special_tokens = True)

    # Compute ROUGE-1 scores
    rouge_scores = rouge.compute(predictions = decoded_preds, 
                                 references = decoded_labels, 
                                 rouge_types = ["rouge1"])["rouge1"]

    # Calculate the mean ROUGE-1 F1 score
    rouge1_f1 = np.mean([score["f"] for score in rouge_scores])

    # Rounds the result to 4 decimal places for cleaner output, and returns it.
    return {"rouge1_f1": round(rouge1_f1, 4)}

# 2. Model Fine-Tuning

In [ ]:
lora_config = LoraConfig(
    task_type = TaskType.SEQ_2_SEQ_LM, 
    inference_mode = False, 
    r = 8, 
    lora_alpha = 32, 
    lora_dropout = 0.3
)

model = AutoModelForSeq2SeqLM.from_pretrained(CFG.model_name)
data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer, model = CFG.model_name)

peft_model = get_peft_model(model, lora_config)

peft_model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    report_to = CFG.report_to,
    output_dir = config["output_dir"],
    num_train_epochs = config["num_train_epochs"],
    per_device_train_batch_size = config["train_batch_size"],
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config["learning_rate"],
    weight_decay = config["weight_decay"],
    # Evaluation
    do_eval = False
)

trainer = Trainer(
    model = peft_model,
    args = training_args,
    train_dataset = tokenized_arxiv["train"],
    data_collator = data_collator,
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train()

test_ver = 5
model_save_path = os.path.join(CFG.model_save_dir, f"test_v{test_ver}")
os.makedirs(model_save_path, exist_ok = True)

trainer.model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

# 3. Evaluation

In [ ]:
import torch

peft_model.eval()

with torch.no_grad():
    outputs = peft_model.generate(input_ids = torch.tensor(tokenized_arxiv['test']['input_ids']), max_new_tokens = 10)
    output_text = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens = True)
print(output_text)

In [ ]:
idx = 0
text = "summarize: " + tokenized_arxiv['test'][idx]['input_text']
actual_text = tokenized_arxiv['test'][idx]['target_text']

tmp = tokenizer(text, return_tensors = 'pt')

In [ ]:
peft_model = peft_model.to("cuda")

peft_model.eval()

with torch.no_grad():
    outputs = peft_model.generate(input_ids = torch.tensor(tokenized_arxiv['test']['input_ids'][0][0]).to("cuda"), max_new_tokens = 10)
    output_text = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens = True)
print(output_text)